# Chương 6: Hệ đa Agent – State, Message, hợp hàm và kiểm tra lỗi

    Hệ điều phối xác định với bốn vai trò; chưa phải nhóm LLM tự lập kế hoạch.
    Chạy trên corpus và FLAN-T5-small cục bộ. Mỗi cell đi kèm điều cần giải thích.

<!-- coursework-vietnamese-guide:agents-essay:v1 -->

### Hướng dẫn đọc cell code 01: Thiết lập một phiên chạy thống nhất

Giới hạn biến môi trường luồng tính toán ở 2, chọn Keras backend TensorFlow, tìm ROOT có src/tieuluan và đổi thư mục chạy về ROOT. Import NumPy/Pandas/Matplotlib và tạo OUT=results/tieuluan/complete/notebooks. Điều kiện môi trường đặt trước khi nạp backend giúp giảm khác biệt số luồng.

**Dữ liệu và kết quả trong phiên chạy.** Các tên được gán ở mức cell gồm `ROOT`, `OUT`. Chúng được giữ trong kernel cho các cell sau; đọc cell từ trên xuống để biết tên nào là dữ liệu vào đã có và tên nào vừa được tạo. Nếu khởi động lại kernel thì cần nạp lại các cell chuẩn bị phụ thuộc trước khi dùng những biến này.

**Cách đọc đầu ra.** `print`/`display` đưa bảng hoặc thông tin ra màn hình; giá trị `return` là kết quả để nơi gọi sử dụng. Cell có mã cập nhật trạng thái, học hoặc ghi file (có thể nằm trong helper chỉ chạy khi được gọi); các thao tác được mô tả ở phần trên. Chỉ đọc notebook không tự thực hiện chúng.


In [1]:
# Cell AGENT-01
import os, sys, json, time, inspect, copy
from pathlib import Path
for key in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS']:
    os.environ[key] = '2'
os.environ['KERAS_BACKEND'] = 'tensorflow'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p/'src/tieuluan').exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
OUT = ROOT/'results/tieuluan/complete/notebooks'
OUT.mkdir(parents=True, exist_ok=True)
print('Project:', ROOT)

Project: E:\WINDOW\Project\intelligent-system-assignment-01


## 1. Mã thực của State, Message và các agent

    State chứa câu hỏi, evidence, graph_context, answer, status và messages.
    Message ghi sender, receiver và payload. Bốn phép chuyển trạng thái hợp thành
    S4 = Verifier ∘ Writer ∘ Graph ∘ Retrieval (S0).
    Cell tiếp theo định nghĩa đầy đủ các lớp, không chỉ import tên.

<!-- coursework-vietnamese-guide:agents-essay:v1 -->

### Hướng dẫn đọc cell code 02: Bốn vai trò và trạng thái thông điệp

Message/State chứa dữ liệu một câu hỏi; RetrievalAgent lấy nguồn, GraphAgent tìm cha nguồn, WriterAgent sinh và VerifierAgent kiểm id/chuỗi rỗng. MultiAgentTeam chạy tuần tự đúng bốn lượt. Graph context được ghi để quan sát nhưng WriterAgent không đưa graph_context vào generate. Verifier đặt semantic_support_checked=False; chưa kiểm chứng nghĩa câu sinh.

**Đầu vào và cách dùng.** Đọc phần định nghĩa/tham số và các lệnh gọi ở cuối cell theo thứ tự. Định nghĩa `def`/`class` đăng ký hàm/lớp để dùng về sau; thân hàm chỉ xử lý dữ liệu khi được gọi. Các giá trị của phiên chạy trước phải có sẵn nếu cell đang gọi lại một helper.

**Hàm, lớp và phép xử lý cần hiểu trong cell này:**

 - **`Message`**: Dataclass lưu sender, receiver và payload. Các tin nhắn là bản ghi Python để quan sát quy trình, không phải dịch vụ gửi tin hay các tiến trình chạy song song.
 - **`State`**: Dataclass lưu câu hỏi/lĩnh vực/report, nguồn, ngữ cảnh graph, câu trả lời, trạng thái và messages. field(default_factory=list) tạo danh sách riêng cho mỗi câu hỏi.
 - **`State.send(self, sender, receiver, **payload)`**: Nhận tên người gửi/nhận và các trường payload rồi append Message vào messages; không thực hiện mạng hay tự gọi agent kế tiếp.
 - **`RetrievalAgent`**: act gọi engine.retrieve với k=3 và report tùy chọn, gán evidence vào State rồi ghi tin retrieval→graph với id nguồn.
 - **`RetrievalAgent.__init__(self, engine)`**: Lưu tham chiếu engine RAG để các lần act dùng cùng bộ truy hồi đã nạp.
 - **`RetrievalAgent.act(self, state)`**: Truy hồi 3 bằng chứng theo question/domain/report, thay state.evidence và gửi danh sách id vào nhật ký.
 - **`GraphAgent`**: act tra nút passage:<id> trong graph theo domain, thêm các nút cha vào graph_context. Ngữ cảnh này dùng để quan sát; WriterAgent hiện chỉ nhận evidence, chưa dùng graph_context để sinh.
 - **`GraphAgent.__init__(self, graphs)`**: Lưu dict domain→graph; nếu domain không có graph thì act chỉ ghi matched=0.
 - **`GraphAgent.act(self, state)`**: Duyệt evidence, tìm các passage hiện hữu, thêm id/parents vào state.graph_context rồi ghi graph→writer.
 - **`WriterAgent`**: Nếu không có nguồn thì đặt insufficient_evidence; nếu có thì generator.generate(question,evidence). Sau đó ghi câu trả lời cho verifier; không tự chọn công cụ mới.
 - **`WriterAgent.__init__(self, generator)`**: Lưu generator cục bộ dùng chung với RAG engine.
 - **`WriterAgent.act(self, state)`**: Sinh câu trả lời từ các evidence đã có hoặc đánh dấu thiếu nguồn, rồi thêm message writer→verifier.
 - **`VerifierAgent`**: Chỉ kiểm id evidence thuộc corpus đúng domain và câu trả lời không rỗng. Hợp lệ vẫn đặt generated_unverified và semantic_support_checked=False; không xác minh mỗi phát biểu được nguồn hỗ trợ.
 - **`VerifierAgent.__init__(self, corpora)`**: Tạo tập id được phép cho từng domain từ corpora; dùng tập để kiểm membership nhanh.
 - **`VerifierAgent.act(self, state)`**: Nếu thiếu/không hợp lệ id hoặc answer rỗng thì thay answer=Insufficient evidence; còn lại ghi generated_unverified và thông báo kết quả kiểm id.
 - **`MultiAgentTeam`**: Ghép bốn vai trò retrieval→graph→writer→verifier chạy tuần tự, mỗi vai trò một lượt. Đây là điều phối hữu hạn với thông điệp rõ, không có vòng lập kế hoạch LLM tự do.
 - **`MultiAgentTeam.__init__(self, engine, graphs)`**: Tạo danh sách bốn đối tượng dùng chung engine/corpora và graph được truyền vào.
 - **`MultiAgentTeam.ask(self, question, domain, report=None)`**: Chặn domain ngoài diabetes/finance và câu hỏi trống, tạo State mới rồi lần lượt gọi act của bốn vai trò. Trả State chứa câu trả lời và messages; không retry.
 - **Helper dùng lại `agent`**: Factory @lru_cache của API: nạp summary thống kê UCI và graph finance rồi tạo ToolAgent. ask định tuyến tất định: thống kê theo một số mẫu chính xác, tính toán với toán hạng bên gọi, truy đồ thị theo company, hoặc RAG. Kiểm citation id tồn tại không kiểm chứng nghĩa câu trả lời.

**Cách đọc đầu ra.** `print`/`display` đưa bảng hoặc thông tin ra màn hình; giá trị `return` là kết quả để nơi gọi sử dụng.


In [2]:
# Cell AGENT-02
"""Bounded, role-based multi-agent demonstration with explicit messages."""
from dataclasses import dataclass, field


@dataclass
class Message:
    sender: str
    receiver: str
    payload: dict


@dataclass
class State:
    question: str
    domain: str
    report: str | None = None
    evidence: list = field(default_factory=list)
    graph_context: list = field(default_factory=list)
    answer: str = ''
    status: str = 'pending'
    messages: list = field(default_factory=list)

    def send(self, sender, receiver, **payload):
        self.messages.append(Message(sender, receiver, payload))


class RetrievalAgent:
    def __init__(self, engine):
        self.engine = engine

    def act(self, state):
        state.evidence = self.engine.retrieve(
            state.question, state.domain, k=3, report=state.report)
        state.send('retrieval', 'graph',
                   ids=[e['id'] for e in state.evidence])


class GraphAgent:
    def __init__(self, graphs):
        self.graphs = graphs

    def act(self, state):
        graph = self.graphs.get(state.domain)
        if graph is not None:
            for item in state.evidence:
                node = 'passage:' + item['id']
                if node in graph:
                    state.graph_context.append({
                        'id': item['id'],
                        'parents': list(graph.predecessors(node))})
        state.send('graph', 'writer',
                   matched=len(state.graph_context))


class WriterAgent:
    def __init__(self, generator):
        self.generator = generator

    def act(self, state):
        if not state.evidence:
            state.status = 'insufficient_evidence'
        else:
            state.answer = self.generator.generate(
                state.question, state.evidence)
        state.send('writer', 'verifier', answer=state.answer)


class VerifierAgent:
    def __init__(self, corpora):
        self.allowed = {d: {x['id'] for x in c}
                        for d, c in corpora.items()}

    def act(self, state):
        ids = [e['id'] for e in state.evidence]
        valid = bool(ids) and all(
            cid in self.allowed[state.domain] for cid in ids)
        if not valid or not state.answer.strip():
            state.answer = 'Insufficient evidence'
            state.status = 'insufficient_evidence'
        else:
            state.status = 'generated_unverified'
        state.send('verifier', 'coordinator', citation_ids_valid=valid,
                   semantic_support_checked=False, status=state.status)


class MultiAgentTeam:
    def __init__(self, engine, graphs):
        self.agents = [RetrievalAgent(engine), GraphAgent(graphs),
                       WriterAgent(engine.generator),
                       VerifierAgent(engine.corpora)]

    def ask(self, question, domain, report=None):
        if domain not in ('diabetes', 'finance'):
            raise ValueError('Unsupported domain')
        if not question.strip():
            raise ValueError('Empty question')
        state = State(question, domain, report)
        for agent in self.agents:  # Four bounded turns; no retry loop.
            agent.act(state)
        return state

## 2. Khởi tạo phụ thuộc

    `rag()` nạp corpus, hai bộ truy hồi và generator; `load_graph` nạp đồ thị
    JSON nén; MultiAgentTeam gắn từng phụ thuộc cho vai trò cần dùng.
    Chỉ dùng một generator chung, không gọi bốn mô hình độc lập.

<!-- coursework-vietnamese-guide:agents-essay:v1 -->

### Hướng dẫn đọc cell code 03: Nối nhóm vai trò với model và graph thật

Nạp graph finance từ JSON.gz, tạo team với rag() đang cache và in tên bốn vai trò. Chỉ finance có graph trong dict này; diabetes vẫn truy hồi/sinh nhưng GraphAgent không có graph để bổ sung cha. agent() được nhập để so công cụ thống kê ở cell cuối.

**Dữ liệu và kết quả trong phiên chạy.** Các tên được gán ở mức cell gồm `graphs`, `team`. Chúng được giữ trong kernel cho các cell sau; đọc cell từ trên xuống để biết tên nào là dữ liệu vào đã có và tên nào vừa được tạo. Nếu khởi động lại kernel thì cần nạp lại các cell chuẩn bị phụ thuộc trước khi dùng những biến này.

**Hàm, lớp và phép xử lý cần hiểu trong cell này:**

 - **Helper dùng lại `MultiAgentTeam`**: Ghép bốn vai trò retrieval→graph→writer→verifier chạy tuần tự, mỗi vai trò một lượt. Đây là điều phối hữu hạn với thông điệp rõ, không có vòng lập kế hoạch LLM tự do.
 - **Helper dùng lại `load_graph`**: Đọc JSON gzip và phục hồi graph NetworkX từ node-link với khóa edges. File cần tồn tại và phù hợp schema.
 - **Helper dùng lại `rag`**: Factory @lru_cache của deployment.chapter04.app: đọc corpus JSONL hai lĩnh vực và tạo RAGEngine với model/cache cục bộ. retrieve tìm nguồn; answer lấy 3 nguồn và sinh, gắn generated_unverified, hoặc insufficient_evidence nếu rỗng.

**Nguồn cài đặt đã đối chiếu:** `deployment.chapter04.app`, `src.chapter04.knowledge`. Các mô tả helper ở trên dựa vào thân hàm/lớp trong dự án; chỉ `import` một tên chưa có nghĩa tên đó đã được gọi trong cell.

**Cách đọc đầu ra.** `print`/`display` đưa bảng hoặc thông tin ra màn hình; giá trị `return` là kết quả để nơi gọi sử dụng.


In [3]:
# Cell AGENT-03
from deployment.chapter04.app import rag, agent
from src.chapter04.knowledge import load_graph
graphs = {'finance':load_graph(
    ROOT/'knowledge_graph/chapter04/finance_graph.json.gz')}
team = MultiAgentTeam(rag(),graphs)
print([type(a).__name__ for a in team.agents])

['RetrievalAgent', 'GraphAgent', 'WriterAgent', 'VerifierAgent']


## 3. Quan sát hợp hàm bằng cách chạy riêng từng agent

    Sau mỗi act, in số bằng chứng, ngữ cảnh đồ thị, câu trả lời, status và
    thông điệp vừa tạo. Retrieval không viết đáp án; Graph bổ sung nguồn cha;
    Writer sinh; Verifier chỉ kiểm tra cấu trúc nguồn, chưa xác minh ngữ nghĩa.

<!-- coursework-vietnamese-guide:agents-essay:v1 -->

### Hướng dẫn đọc cell code 04: Quan sát từng bước của một câu hỏi

Tạo State hỏi thay đổi doanh thu rồi tự lần lượt gọi worker.act, ghi số evidence/graph, answer/status và in message cuối mỗi vai trò. assert đủ bốn message và verifier không kiểm semantic support. Câu hỏi không khóa report nên có thể truy nguồn nhiều báo cáo, cần đọc nguồn trước khi diễn giải số sinh.

**Dữ liệu và kết quả trong phiên chạy.** Các tên được gán ở mức cell gồm `state`, `trace`. Chúng được giữ trong kernel cho các cell sau; đọc cell từ trên xuống để biết tên nào là dữ liệu vào đã có và tên nào vừa được tạo. Nếu khởi động lại kernel thì cần nạp lại các cell chuẩn bị phụ thuộc trước khi dùng những biến này.

**Hàm, lớp và phép xử lý cần hiểu trong cell này:**

 - **Helper dùng lại `State`**: Dataclass lưu câu hỏi/lĩnh vực/report, nguồn, ngữ cảnh graph, câu trả lời, trạng thái và messages. field(default_factory=list) tạo danh sách riêng cho mỗi câu hỏi.

**Cách đọc đầu ra.** `print`/`display` đưa bảng hoặc thông tin ra màn hình; giá trị `return` là kết quả để nơi gọi sử dụng. `assert` chỉ dừng khi điều kiện cụ thể trong mã sai; các ca đi qua không phải bằng chứng rằng mọi đầu vào đều đúng.


In [4]:
# Cell AGENT-04
state = State('What is the percentage change in revenue?','finance')
trace = []
for worker in team.agents:
    worker.act(state)
    trace.append({'agent':type(worker).__name__,
        'evidence':len(state.evidence),'graph':len(state.graph_context),
        'answer':state.answer,'status':state.status})
    message = state.messages[-1]
    print(message.sender,'->',message.receiver,message.payload)
display(pd.DataFrame(trace))
assert len(state.messages)==4
assert not state.messages[-1].payload['semantic_support_checked']

retrieval -> graph {'ids': ['ETR/2004/page_213.pdf:d81f966dc19b::text_6', 'ETR/2008/page_377.pdf:26c3c754ec34::text_6', 'ETR/2011/page_341.pdf:ce32d713fc20::text_2']}
graph -> writer {'matched': 3}


writer -> verifier {'answer': 'unknown'}
verifier -> coordinator {'citation_ids_valid': True, 'semantic_support_checked': False, 'status': 'generated_unverified'}


,agent,evidence,graph,answer,status
0,RetrievalAgent,3,0,,pending
1,GraphAgent,3,3,,pending
2,WriterAgent,3,3,unknown,pending
3,VerifierAgent,3,3,unknown,generated_unverified


## 4. Kiểm tra các đường lỗi

    `ask` chặn domain sai và câu hỏi rỗng. Khi report không tồn tại, truy hồi
    trả rỗng, Writer không sinh và Verifier trả insufficient_evidence.
    Điểm dừng: mỗi agent chạy một lần, không vòng lặp đến khi đồng ý.

<!-- coursework-vietnamese-guide:agents-essay:v1 -->

### Hướng dẫn đọc cell code 05: Thiếu nguồn và đầu vào không hợp lệ

team.ask với report không tồn tại phải trả insufficient_evidence. Hai ca câu hỏi rỗng/domain lạ phải ValueError; else raise để phát hiện không chặn. Đây là kiểm luồng điều phối hữu hạn, không đánh giá độ chính xác mọi câu hỏi tài chính.

**Dữ liệu và kết quả trong phiên chạy.** Các tên được gán ở mức cell gồm `missing`. Chúng được giữ trong kernel cho các cell sau; đọc cell từ trên xuống để biết tên nào là dữ liệu vào đã có và tên nào vừa được tạo. Nếu khởi động lại kernel thì cần nạp lại các cell chuẩn bị phụ thuộc trước khi dùng những biến này.

**Cách đọc đầu ra.** `print`/`display` đưa bảng hoặc thông tin ra màn hình; giá trị `return` là kết quả để nơi gọi sử dụng. `assert` chỉ dừng khi điều kiện cụ thể trong mã sai; các ca đi qua không phải bằng chứng rằng mọi đầu vào đều đúng.


In [5]:
# Cell AGENT-05
missing = team.ask('Find revenue','finance',report='__missing_report__')
print('Không có nguồn:',missing.status,missing.answer)
assert missing.status=='insufficient_evidence'
for question, domain in [('', 'finance'), ('test','invalid')]:
    try:
        team.ask(question,domain)
    except ValueError as exc:
        print('Đầu vào bị chặn:',exc)
    else:
        raise AssertionError('Đầu vào sai không bị chặn')

Không có nguồn: insufficient_evidence Insufficient evidence
Đầu vào bị chặn: Empty question
Đầu vào bị chặn: Unsupported domain


## 5. Vì sao nguồn tồn tại chưa chứng minh câu trả lời đúng?

    Chạy một câu hỏi đếm qua bộ sinh rồi đối chiếu công cụ statistics của
    ToolAgent. Chỉ statistics đọc số từ bảng thật; không dùng câu trả lời
    sai của generator như dữ kiện. Đây là phép thử giới hạn Verifier.

<!-- coursework-vietnamese-guide:agents-essay:v1 -->

### Hướng dẫn đọc cell code 06: So sinh tự do với thống kê xác định

Cùng câu hỏi số lượt nhập viện, nhóm RAG sinh từ tài liệu trong khi agent().ask(task=statistics) đọc bộ đếm summary. In answer/status/id-valid và lưu agents_demo.json gồm trace/messages/count. Giá trị computed dựa bộ đếm có sẵn; id-valid của câu sinh không chứng minh số lượng đúng.

**Dữ liệu và kết quả trong phiên chạy.** Các tên được gán ở mức cell gồm `question`, `generated`, `computed`, `payload`. Chúng được giữ trong kernel cho các cell sau; đọc cell từ trên xuống để biết tên nào là dữ liệu vào đã có và tên nào vừa được tạo. Nếu khởi động lại kernel thì cần nạp lại các cell chuẩn bị phụ thuộc trước khi dùng những biến này.

**Hàm, lớp và phép xử lý cần hiểu trong cell này:**

 - **Helper dùng lại `agent`**: Factory @lru_cache của API: nạp summary thống kê UCI và graph finance rồi tạo ToolAgent. ask định tuyến tất định: thống kê theo một số mẫu chính xác, tính toán với toán hạng bên gọi, truy đồ thị theo company, hoặc RAG. Kiểm citation id tồn tại không kiểm chứng nghĩa câu trả lời.

**Cách đọc đầu ra.** `print`/`display` đưa bảng hoặc thông tin ra màn hình; giá trị `return` là kết quả để nơi gọi sử dụng. Cell có mã cập nhật trạng thái, học hoặc ghi file (có thể nằm trong helper chỉ chạy khi được gọi); các thao tác được mô tả ở phần trên. Chỉ đọc notebook không tự thực hiện chúng.


In [6]:
# Cell AGENT-06
question = 'How many encounters are in the dataset?'
generated = team.ask(question,'diabetes')
computed = agent().ask(question,'diabetes',task='statistics')
print('Sinh:',generated.answer,generated.status)
print('Thống kê:',computed['answer'],computed['status'])
print('Nguồn id hợp lệ:',
      generated.messages[-1].payload['citation_ids_valid'])
payload = {'finance_trace':trace,
    'messages':[vars(m) for m in state.messages],
    'generated_count':generated.answer,
    'computed_count':computed['answer'],
    'missing_status':missing.status}
(OUT/'agents_demo.json').write_text(
    json.dumps(payload,ensure_ascii=False,indent=2),encoding='utf8')

Sinh: 2 generated_unverified
Thống kê: 101766 computed
Nguồn id hợp lệ: True


1440

## Câu hỏi trình bày

    Hãy giải thích riêng State.send, RetrievalAgent.act, GraphAgent.act,
    WriterAgent.act, VerifierAgent.act và MultiAgentTeam.ask. Chỉ ra trường
    State nào bị đổi ở từng bước, Message gửi cho ai và điều kiện dừng/lỗi.
    Tại sao không được đổi generated_unverified thành verified chỉ vì id hợp lệ?